[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Реализуйте учебный слой **Mixture of Experts (MoE)** в стиле sparse Transformer. Вместо одного MLP слой содержит несколько experts, а маленький router выбирает для каждого токена только `top_k` наиболее подходящих. Так ёмкость параметров может расти быстрее, чем вычисления на токен.

## Router, experts и смешивание

Вход `(B,S,D)` удобно мысленно рассматривать как `N=B*S` независимых токенов. Router выдаёт logits `(N,E)`, где E — число экспертов. `topk` выбирает индексы `(N,K)`, а softmax **только по выбранным logits** создаёт веса, суммирующиеся в 1 для каждого токена. Итог токена — взвешенная сумма K expert outputs формы `(D,)`.

Например, при четырёх experts и `top_k=2` router может выбрать experts 1 и 3 с весами `[0.8,0.2]`; output равен `0.8*expert1(x)+0.2*expert3(x)`. При `top_k=1` выбранный expert получает вес 1. Router и experts обучаются совместно через градиент по выбранным путям.

> Учебная реализация с Python-циклами может вычислительно не быть sparse и быстрой. Production MoE группирует токены по experts, выполняет batched kernels и обменивает токены между устройствами.

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

## Что остаётся за границей задания

Без дополнительных мер router может отправлять почти все токены одному expert. Реальные системы используют load-balancing auxiliary loss, capacity factor, drop/overflow policy и иногда router z-loss. Для прохождения упражнения эти механизмы добавлять не нужно.

## План реализации

1. Создайте router и `ModuleList` из `num_experts` MLP одинаковой архитектуры.
2. Flatten только batch/sequence, сохранив D.
3. Получите top-k logits/indices и нормализуйте выбранные logits.
4. Накопите взвешенные expert outputs для каждого токена и восстановите `(B,S,D)`.

## Быстрые проверки

- Gating weights каждого токена суммируются в 1.
- При `top_k=1` output совпадает с выходом единственного выбранного expert.
- При одинаковых expert outputs результат не зависит от router weights.
- Output shape совпадает с input, backward достигает router и выбранных experts.

## Частые ошибки

- Делать softmax по всем experts до top-k: веса выбранных тогда не суммируются в 1 без повторной нормализации.
- Использовать один и тот же MLP-объект несколько раз вместо независимых modules.
- Потерять соответствие token index ↔ выбранный expert при scatter/add.
- Обещать реальное sparse-ускорение от наивного цикла.

## Материалы для глубокого изучения

- [Switch Transformers](https://arxiv.org/abs/2101.03961) — top-1 routing, capacity и load balancing.
- [Mixtral of Experts](https://arxiv.org/abs/2401.04088) — sparse top-2 MoE в современной LLM.
- [PyTorch topk](https://docs.pytorch.org/docs/stable/generated/torch.topk.html) — контракт выбора значений и индексов.

## Вопросы для самопроверки

1. Почему softmax применяется после top-k именно к выбранным logits?
2. Чем параметрическая ёмкость MoE отличается от вычислений на один токен?
3. Зачем production MoE нужен load-balancing loss?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        pass  # router + experts

    def forward(self, x):
        pass  # route tokens to top-k experts

In [ ]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('moe')